# 🧪 RUPS Pilot Study — Clean Notebook v2
### *Reasoning Under Pressure: Structured Domain Failures in Frontier LLMs*

**Status:** Pilot complete ✅ | Scaffolds validated (v2) ✅ | Ready for RUPS-300 ✅

| Step | Description | Status |
|------|-------------|--------|
| 1 | Setup & gateway | ✅ |
| 2 | Load 20 pilot problems | ✅ |
| 3 | Run 300 calls (5 models × 3 conditions) | ✅ |
| 4 | Auto-annotate with LLM-as-judge | ✅ |
| 5 | Validate & fix scaffolds (v1 → v2) | ✅ |
| 6 | Final analysis & GO decision | ✅ GO |

**Key pilot findings:**
- All 5 failure modes confirmed across 18/20 problems
- Gemini-2.5-flash most vulnerable (80% baseline failure rate)
- GPT-5.4-mini most robust (5% baseline failure rate)
- F4 scaffold v2: +44% recovery | F1/F2/F3/F5 scaffold v2: improved in 9/10 problems
- Decision: **GO → proceed to RUPS-300 full benchmark**

---

## 📦 Cell 1 — Install & imports

In [ ]:
!pip install aiohttp nest_asyncio --quiet

import requests, aiohttp, asyncio, time, json, csv, os
from datetime import datetime
from collections import Counter
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

print('✅ Ready')

## 🔧 Cell 2 — LLM Gateway

In [ ]:
class TokenService:
    """
    Authentication service for LLM gateway.

    To reproduce experiments, replace this class with direct API calls
    to your preferred LLM provider using the model names in Table 4.

    Supported alternatives:
      - OpenAI:    openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
      - Anthropic: anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
      - Groq:      groq.Groq(api_key=os.environ["GROQ_API_KEY"])

    See README.md for full reproduction instructions.
    """
    def __init__(self):
        # Set via environment variable: export GATEWAY_AUTH_URL="your_url"
        self.url     = os.environ.get("GATEWAY_AUTH_URL", "YOUR_GATEWAY_AUTH_URL")
        self.payload = os.environ.get("GATEWAY_CREDENTIALS", "YOUR_CREDENTIALS")
        self.headers = {'Content-Type': 'text/plain'}

    def get_token(self):
        for attempt in range(4):
            try:
                response = requests.post(self.url, headers=self.headers,
                                         data=self.payload, timeout=120)
                if response.status_code == 200:
                    data = response.json()
                    return f"Bearer {data['access_token']}", 200
                print(f"Token API failed: {response.status_code}")
            except Exception as e:
                print("Token Fetch Error:", str(e))
            time.sleep(1)
        return None, 408


class AsyncLLM:
    def __init__(self, environment="QA"):
        # Set via environment variable: export GATEWAY_URL="your_url"
        self.url = os.environ.get(
            "GATEWAY_URL",
            "YOUR_GATEWAY_URL/v1/chat/completion"
        )
        self.token   = None
        self.headers = {'Content-Type': 'application/json', 'Authorization': ''}

    def fetch_token(self):
        ts = TokenService()
        token, status = ts.get_token()
        if status == 200:
            self.token = token
            self.headers['Authorization'] = token
        else:
            raise Exception("Failed to fetch token — check GATEWAY_AUTH_URL and GATEWAY_CREDENTIALS env vars")

    def get_provider(self, engine):
        if "gemini" in engine.lower() or "imagen" in engine.lower():
            return "GoogleBard"
        elif "claude" in engine.lower():
            return "Anthropic"
        return None

    def build_payload(self, engine, messages, temperature, max_tokens, custom_identifier):
        payload = {
            "ModelToUse": engine,
            "AdditionalRequestParameters": {
                "Temperature": temperature,
                "MaxTokens": max_tokens,
                "TopProbabilities": 0.5,
                "PresencePenalty": 0,
                "FrequencyPenalty": 0
            },
            "CustomIdentifier": custom_identifier or "rups-experiment",
            "Messages": [
                {"Role": m.get("role", m.get("Role", "user")).capitalize(),
                 "Content": m.get("content", m.get("Content", ""))}
                for m in messages
            ]
        }
        if engine.startswith("gpt-5"):
            payload["AdditionalRequestParameters"].pop("StopSequences", None)
        provider = self.get_provider(engine)
        if provider:
            payload["LLMProviderType"] = provider
        return payload

    async def infer(self, session, engine, messages, temperature, max_tokens,
                    custom_identifier, semaphore):
        payload = self.build_payload(engine, messages, temperature, max_tokens, custom_identifier)
        async with semaphore:
            for attempt in range(3):
                try:
                    async with session.post(
                        self.url, headers=self.headers, json=payload,
                        timeout=aiohttp.ClientTimeout(total=240)
                    ) as resp:
                        if resp.status == 200:
                            return await resp.json(), 200
                        elif resp.status == 401:
                            self.fetch_token()
                            continue
                        else:
                            text = await resp.text()
                            return {"error": text}, resp.status
                except asyncio.TimeoutError:
                    print(f"  ⏱️ Timeout (attempt {attempt+1}/3)")
                    await asyncio.sleep(2)
                except Exception as e:
                    print(f"  ⚠️ Error (attempt {attempt+1}/3): {e}")
                    await asyncio.sleep(2)
        return {"error": "Max retries exceeded"}, 500

    def extract_text(self, response):
        try:
            return response["Messages"][0]["Message"]["Content"]
        except (KeyError, IndexError, TypeError):
            return None


print('✅ LLM Gateway class loaded')
print('   Note: Set GATEWAY_AUTH_URL and GATEWAY_CREDENTIALS environment variables')
print('   See README.md for alternative provider setup (OpenAI/Anthropic/Groq)')


## 🔑 Cell 3 — Init gateway & set save directory

In [ ]:
llm = AsyncLLM(environment="QA")
llm.fetch_token()
print(f'✅ Token: {llm.token[:35]}...' if llm.token else '❌ Token failed')

SAVE_DIR = './results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Results directory: {SAVE_DIR}')

## 📋 Cell 4 — Load pilot problems
> Make sure `rups296_benchmark.json` is in the same directory as this notebook

In [ ]:
#problems = json.load(open('rups296_benchmark.json'))
problems = json.load(open('rups296_benchmark.json'))

print(f'✅ {len(problems)} problems loaded')
print(f'   Failure modes : {dict(Counter(p["failure_mode"] for p in problems))}')
print(f'   Domains       : {dict(Counter(p["domain"] for p in problems))}')
print(f'   With adversarial: {sum(1 for p in problems if p.get("adversarial_version"))}')

## ⚙️ Cell 5 — Config: models, conditions & scaffolds (v2)
> Scaffolds v2 — validated and fixed after pilot diagnostic (9/10 problems improved)

In [ ]:
# ── Models ────────────────────────────────────────────────────
MODELS = [
    'gpt-4o',            # OpenAI GPT-4 — primary baseline
    'gpt-4o-mini',       # OpenAI GPT-4 — lightweight
    'claude-sonnet-4.6', # Anthropic — cross-company
    'gemini-2.5-flash',  # Google — cross-family
    'gpt-5.4-mini',      # OpenAI GPT-5 — next-gen comparison
]

CONDITIONS   = ['baseline', 'cot', 'context_scaffold']
TEMPERATURE  = 0.1
MAX_TOKENS   = 1200   # increased from 800 — fixes Gemini truncation
CONCURRENCY  = 8

# ── Scaffolds v2 (validated after pilot diagnostic) ───────────
# v1 problems fixed:
#   F1: was causing models to incorrectly define fields
#   F2: was making models focus on order, miss trend implication
#   F3: was making models invent unit mismatches
#   F5: was causing models to JUSTIFY the spurious feature
#   F4: unchanged — worked perfectly (+44% recovery in v1)

SCAFFOLDS = {
    'F1': (
        'INSTRUCTION: Field and column names in the following data are precise '
        'technical terms — each has a specific meaning that may not be obvious '
        'from its value alone. Do not guess a field\'s meaning from its number. '
        'If a field name is abbreviated or technical, treat it as domain-specific '
        'terminology before drawing conclusions.'
    ),
    'F2': (
        'INSTRUCTION: The following records are time-ordered. The TREND and '
        'DIRECTION of change matters more than any single value. Before answering, '
        'identify whether each measured variable is improving, worsening, or stable '
        'over time, and what the combined trajectory implies.'
    ),
    'F3': (
        'INSTRUCTION: The following data may contain values in different units or '
        'scales. Before making any comparison or calculation, convert all values '
        'to the same unit. State the unit you are working in at the start of '
        'your answer.'
    ),
    'F4': (
        'INSTRUCTION: The following data contains both positive (present) and '
        'negative (absent/denied) findings. Negative findings are analytically '
        'significant. List all negations explicitly before drawing any conclusions.'
    ),
    'F5': (
        'INSTRUCTION: Assess this individual based solely on their specific '
        'profile values. Do not let category labels (job title, loan purpose, '
        'demographic fields) override what the individual\'s actual metrics show. '
        'Group-level statistics do not predict individual outcomes — focus on '
        'the numbers, not the labels.'
    ),
}

def build_prompt(problem, condition):
    ctx, q = problem['context'], problem['question']
    if condition == 'baseline':
        return f"{ctx}\n\nQuestion: {q}"
    elif condition == 'cot':
        return f"Think carefully step by step before your final answer.\n\n{ctx}\n\nQuestion: {q}"
    elif condition == 'context_scaffold':
        return f"{SCAFFOLDS[problem['failure_mode']]}\n\n{ctx}\n\nQuestion: {q}"

total = len(problems) * len(MODELS) * len(CONDITIONS)
print(f'✅ Config ready (scaffolds v2)')
print(f'   {len(problems)} problems × {len(MODELS)} models × {len(CONDITIONS)} conditions = {total} calls')

## 🚀 Cell 6 — Run experiment
> ~8–12 minutes. Do not restart kernel after this.

In [ ]:
start_time = datetime.now()

async def run_one(session, semaphore, problem, model, condition):
    prompt = build_prompt(problem, condition)
    t0 = time.time()
    raw, status = await llm.infer(
        session, model,
        [{'role': 'user', 'content': prompt}],
        temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        custom_identifier=f"rups-{problem['id']}-{model}-{condition}",
        semaphore=semaphore
    )
    return {
        'id':                 problem['id'],
        'domain':             problem['domain'],
        'failure_mode':       problem['failure_mode'],
        'failure_name':       problem['failure_name'],
        'structural_trigger': problem['structural_trigger'],
        'model':              model,
        'condition':          condition,
        'http_status':        status,
        'elapsed_sec':        round(time.time() - t0, 2),
        'response':           llm.extract_text(raw) if status == 200 else None,
        'error':              raw.get('error') if status != 200 else None,
    }

async def run_all(problem_set=None, condition_set=None):
    """Run experiment. Pass subsets to rerun specific problems/conditions."""
    p_set = problem_set or problems
    c_set = condition_set or CONDITIONS
    semaphore = asyncio.Semaphore(CONCURRENCY)
    batch = []
    async with aiohttp.ClientSession() as session:
        coros = [
            run_one(session, semaphore, p, m, c)
            for p in p_set for m in MODELS for c in c_set
        ]
        total_c = len(coros)
        print(f'🚀 {total_c} calls (concurrency={CONCURRENCY})...\n')
        for coro in asyncio.as_completed(coros):
            r = await coro
            batch.append(r)
            done = len(batch)
            pct  = int(100 * done / total_c)
            bar  = '█' * (pct // 5) + '░' * (20 - pct // 5)
            icon = '❌' if r['error'] else '✅'
            print(f'\r  [{bar}] {done}/{total_c} ({pct}%)  '
                  f'{icon} {r["model"][:20]:<20} {r["condition"]:<18} {r["id"][:14]}',
                  end='', flush=True)
    print()
    return batch

results = asyncio.run(run_all())

elapsed = (datetime.now() - start_time).seconds
errors  = [r for r in results if r['error']]
print(f'\n✅ Done in {elapsed}s | Total: {len(results)} | Errors: {len(errors)}')
for e in errors[:3]:
    print(f'   └─ {e["id"]} / {e["model"]}: {str(e["error"])[:80]}')

## 💾 Cell 7 — Save raw results

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M')

raw_json = f'{SAVE_DIR}/results/experiment_results_{ts}.json'
raw_csv  = f'{SAVE_DIR}/results/experiment_results_{ts}.csv'

with open(raw_json, 'w') as f:
    json.dump(results, f, indent=2)

with open(raw_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f'✅ Raw results saved:')
print(f'   {raw_json}')
print(f'   {raw_csv}')

## 🤖 Cell 8 — Auto-annotate with LLM-as-judge
> Uses GPT-4o to score each response 0/1/2 against the gold answer
> Cite: *LLM-as-judge methodology (Zheng et al., 2023)*
>
> ~15 minutes for 300 responses

In [ ]:
JUDGE_MODEL       = 'gpt-4o'
JUDGE_TEMP        = 0.0
JUDGE_TOKENS      = 300
JUDGE_CONCURRENCY = 5

JUDGE_PROMPT = """You are an expert annotator for an AI research study on LLM reasoning failures.
Score whether a model response exhibits a specific reasoning failure mode.

## Failure Mode
{failure_mode}: {failure_name}
Trigger: {structural_trigger}
Failure fires when: {gold_failure_label}

## Gold Answer
{gold_answer}

## Model Response
{response}

## Scoring
0 = Failure NOT exhibited (model answered correctly)
1 = Partial failure (partially correct, missed key element)
2 = Failure CLEARLY exhibited (fell for the trigger in predicted way)

Respond ONLY with JSON:
{{"score": <0/1/2>, "reason": "<one sentence>", "key_error": "<specific error or null>"}}"""


async def judge_one(session, semaphore, result, prob_map):
    prob = prob_map.get(result['id'])
    if not prob or not result.get('response'):
        return {**result, 'judge_score': None, 'judge_reason': None,
                'judge_key_error': None, 'judge_error': 'no response'}

    prompt = JUDGE_PROMPT.format(
        failure_mode       = result['failure_mode'],
        failure_name       = result['failure_name'],
        structural_trigger = result['structural_trigger'],
        gold_failure_label = prob['gold_failure_label'],
        gold_answer        = prob['gold_answer'],
        response           = result['response'][:2000]
    )
    raw, status = await llm.infer(
        session, JUDGE_MODEL,
        [{'role': 'user', 'content': prompt}],
        temperature=JUDGE_TEMP, max_tokens=JUDGE_TOKENS,
        custom_identifier=f"judge-{result['id']}-{result['model']}-{result['condition']}",
        semaphore=semaphore
    )
    if status != 200:
        return {**result, 'judge_score': None, 'judge_reason': None,
                'judge_key_error': None, 'judge_error': raw.get('error')}

    raw_text = llm.extract_text(raw) or ''
    try:
        clean = raw_text.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
        parsed = json.loads(clean)
        return {**result,
                'judge_score':     int(parsed.get('score', -1)),
                'judge_reason':    parsed.get('reason', ''),
                'judge_key_error': parsed.get('key_error'),
                'judge_error':     None}
    except Exception as e:
        return {**result, 'judge_score': -1, 'judge_reason': f'parse error: {e}',
                'judge_key_error': raw_text[:100], 'judge_error': None}


async def run_judge(result_set):
    prob_map  = {p['id']: p for p in problems}
    to_judge  = [r for r in result_set if not r.get('error') and r.get('response')]
    semaphore = asyncio.Semaphore(JUDGE_CONCURRENCY)
    judged    = []
    print(f'🤖 Judging {len(to_judge)} responses with {JUDGE_MODEL}...')
    async with aiohttp.ClientSession() as session:
        coros = [judge_one(session, semaphore, r, prob_map) for r in to_judge]
        for coro in asyncio.as_completed(coros):
            j = await coro
            judged.append(j)
            done = len(judged)
            pct  = int(100 * done / len(coros))
            bar  = '█' * (pct // 5) + '░' * (20 - pct // 5)
            print(f'\r  [{bar}] {done}/{len(coros)} ({pct}%)  '
                  f'score={j.get("judge_score")}  '
                  f'{j["model"][:18]:<18} {j["condition"]:<18} {j["id"][:12]}',
                  end='', flush=True)
            await asyncio.sleep(0.1)
    print()
    return judged


results/judged_results = asyncio.run(run_judge(results))

valid  = [j for j in results/judged_results if j.get('judge_score', -1) >= 0]
errors = [j for j in results/judged_results if j.get('judge_score', -1) < 0]
dist   = Counter(j['judge_score'] for j in valid)
print(f'\n✅ Judging complete: {len(valid)} scored, {len(errors)} errors')
print(f'   0 (no failure):    {dist[0]:3d} ({100*dist[0]//max(len(valid),1)}%)')
print(f'   1 (partial):       {dist[1]:3d} ({100*dist[1]//max(len(valid),1)}%)')
print(f'   2 (clear failure): {dist[2]:3d} ({100*dist[2]//max(len(valid),1)}%)')

# Save
ts2 = datetime.now().strftime('%Y%m%d_%H%M')
judged_json = f'{SAVE_DIR}/results/judged_results_{ts2}.json'
judged_csv  = f'{SAVE_DIR}/results/judged_results_{ts2}.csv'
with open(judged_json, 'w') as f:
    json.dump(results/judged_results, f, indent=2)
with open(judged_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=results/judged_results[0].keys())
    writer.writeheader()
    writer.writerows(results/judged_results)
print(f'\n💾 Judged results saved:')
print(f'   {judged_json}')
print(f'   {judged_csv}')

## 📊 Cell 9 — Full analysis & paper results

In [ ]:
# ── TABLE 3 FIX — handles missing model/problem combinations ──
base  = df[df['condition'] == 'baseline']
piv   = base.pivot_table('failed', 'id', 'model', 'mean').round(2)
n_mod = df['model'].nunique()

# Fill NaN with 0 before converting (missing = no failure recorded)
piv_int = piv.fillna(0).astype(int)
piv_int['all_fail'] = (piv_int.sum(axis=1) == n_mod).astype(int)
piv_int['any_fail'] = (piv_int.sum(axis=1) > 0).astype(int)

print('=== TABLE 3: CROSS-MODEL CONSISTENCY (baseline only) ===')
print(f'  All {n_mod} models failed on same problem: {piv_int["all_fail"].sum()}/{len(piv_int)}')
print(f'  At least 1 model failed:                  {piv_int["any_fail"].sum()}/{len(piv_int)}')

# Bonus — most consistent failure problems
print(f'\n  Problems failed by 4+ models (strongest cross-model signal):')
piv_int['fail_count'] = piv_int[list(df["model"].unique())].sum(axis=1)
top = piv_int[piv_int['fail_count'] >= 4].sort_values('fail_count', ascending=False)
print(f'  Count: {len(top)}')
for pid, row in top.head(10).iterrows():
    fm = df[df['id']==pid]['failure_mode'].iloc[0] if len(df[df['id']==pid]) > 0 else '?'
    print(f'    {pid:<20} {fm}  failed by {int(row["fail_count"])}/5 models')

In [ ]:
df = pd.DataFrame([j for j in results/judged_results if j.get('judge_score', -1) >= 0])
df['failed'] = (df['judge_score'] >= 1).astype(int)

print('=' * 65)
print('  RUPS EXP-300 — FULL RESULTS')
print('=' * 65)

print('\n=== TABLE 1: FAILURE RATE BY MODEL × CONDITION (%) ===')
print('Lower is better. This is Table 1 in your paper.')
t1 = df.pivot_table('failed', 'model', 'condition', 'mean').round(3) * 100
print(t1.to_string())

print('\n=== TABLE 2: SCAFFOLD RECOVERY RATE BY FAILURE MODE ===')
print('This is Table 2 in your paper.')
for fm in sorted(df['failure_mode'].unique()):
    b   = df[(df['failure_mode']==fm) & (df['condition']=='baseline')]['failed'].mean()
    c   = df[(df['failure_mode']==fm) & (df['condition']=='cot')]['failed'].mean()
    s   = df[(df['failure_mode']==fm) & (df['condition']=='context_scaffold')]['failed'].mean()
    rec = (b - s) / b * 100 if b > 0 else 0
    fn  = df[df['failure_mode']==fm]['failure_name'].iloc[0][:22]
    bar = '█' * int(abs(rec) / 10)
    sign = '+' if rec > 0 else ''
    print(f'  {fm} {fn:<24} base={b:.0%}  cot={c:.0%}  '
          f'scaffold={s:.0%}  recovery={sign}{rec:.0f}%  {bar}')

print('\n=== TABLE 3: CROSS-MODEL CONSISTENCY (baseline only) ===')
print('Supports Claim C4 — failures are architecture-level.')
base  = df[df['condition'] == 'baseline']
piv   = base.pivot_table('failed', 'id', 'model', 'mean').round(0).astype(int)
n_mod = df['model'].nunique()
piv['all_fail'] = (piv.sum(axis=1) == n_mod).astype(int)
piv['any_fail'] = (piv.sum(axis=1) > 0).astype(int)
print(f'  All {n_mod} models failed on same problem: {piv["all_fail"].sum()}/{len(piv)}')
print(f'  At least 1 model failed:                  {piv["any_fail"].sum()}/{len(piv)}')

print('\n=== TOP FAILURE ERRORS (judge explanations) ===')
key_errors = [j['judge_key_error'] for j in results/judged_results
              if j.get('judge_score', 0) >= 1 and j.get('judge_key_error')]
for i, err in enumerate(key_errors[:8], 1):
    print(f'  {i}. {str(err)[:90]}')

print('\n=== GO / NO-GO DECISION ===')
distinct = df[df['failed'] == 1]['failure_mode'].nunique()
print(f'  Distinct failure modes confirmed: {distinct}/5')
if distinct >= 3:
    print('  ✅ GO — proceed to RUPS-300 full benchmark (Week 3-4)')
else:
    print('  ⚠️  REVISE — fewer than 3 modes confirmed')

## 🔍 Cell 10 — Response browser
> Read individual responses for any problem/model/condition

In [ ]:
# ── Edit these two values to browse ──────────────────────────
BROWSE_ID    = 'FIN-F1-001'
BROWSE_MODEL = 'gpt-4o'
# ─────────────────────────────────────────────────────────────

prob = next((p for p in problems if p['id'] == BROWSE_ID), None)
if prob:
    print(f"{'='*70}")
    print(f"PROBLEM : {prob['id']}  |  {prob['failure_mode']} — {prob['failure_name']}")
    print(f"Trigger : {prob['structural_trigger']}")
    print(f"{'='*70}")
    print(f"\nCONTEXT:\n{prob['context']}")
    print(f"\nQUESTION:\n{prob['question']}")
    print(f"\nGOLD ANSWER:\n{prob['gold_answer']}")
    print(f"\nFAILURE FIRES WHEN:\n{prob['gold_failure_label']}")
    for cond in CONDITIONS:
        row = next((r for r in results/judged_results
                    if r['id']==BROWSE_ID and r['model']==BROWSE_MODEL
                    and r['condition']==cond), None)
        print(f"\n{'─'*70}")
        print(f"CONDITION: {cond.upper()}  |  Judge score: {row.get('judge_score') if row else 'N/A'}")
        print(f"Judge reason: {row.get('judge_reason','') if row else ''}")
        print(f"{'─'*70}")
        print(row['response'] if row and row.get('response') else '[No response]')

## ⚔️ Cell 11 — Adversarial pair test
> Confirm the structural trigger caused the failure (not just problem difficulty)

In [ ]:
# ── Edit these values ─────────────────────────────────────────
ADV_ID    = 'FIN-F1-001'   # must have adversarial_version in rups296_benchmark.json
ADV_MODEL = 'gpt-4o'
# ─────────────────────────────────────────────────────────────

prob = next((p for p in problems if p['id'] == ADV_ID), None)
if not prob or not prob.get('adversarial_version'):
    print(f'No adversarial version for {ADV_ID}')
else:
    adv_prompt = f"{prob['adversarial_version']}\n\nQuestion: {prob['question']}"

    async def run_adv():
        sem = asyncio.Semaphore(1)
        async with aiohttp.ClientSession() as session:
            raw, status = await llm.infer(
                session, ADV_MODEL,
                [{'role': 'user', 'content': adv_prompt}],
                temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                custom_identifier='rups-adversarial', semaphore=sem
            )
            return llm.extract_text(raw) if status == 200 else f'ERROR: {raw}'

    adv_resp = asyncio.run(run_adv())
    orig = next((r for r in results/judged_results
                 if r['id']==ADV_ID and r['model']==ADV_MODEL
                 and r['condition']=='baseline'), None)

    print(f"{'='*70}")
    print(f"ADVERSARIAL PAIR — {ADV_ID} / {ADV_MODEL}")
    print(f"Failure: {prob['failure_mode']} — {prob['failure_name']}")
    print(f"{'='*70}")
    print(f"\n{'─'*70}\nORIGINAL (structured trigger present)")
    print(f"Judge score: {orig.get('judge_score') if orig else 'N/A'}")
    print(f"{'─'*70}")
    print(orig['response'] if orig and orig.get('response') else '[not found]')
    print(f"\n{'─'*70}\nADVERSARIAL (trigger removed — plain language)\n{'─'*70}")
    print(adv_resp)
    print(f"\n>>> Adversarial CORRECT + Original WRONG → {prob['failure_mode']} CONFIRMED ✅")
    print(f"    Both wrong → problem difficulty, not trigger")
    print(f"    Both correct → trigger didn't cause failure for this model")

---
## 📌 Pilot complete — next steps

**Results summary (from actual pilot run 2026-05-09):**
- 300 responses collected, 300 auto-annotated (0 errors)
- Score distribution: 62% no failure | 15% partial | 22% clear failure
- All 5 failure modes confirmed — **GO decision**
- Scaffolds v2 validated — 9/10 degraded problems improved

**Week 3–4: Build RUPS-300**
- Scale from 20 → 300 problems (100 per domain)
- Same structure: (context, question, gold answer, failure label, adversarial)
- Release on HuggingFace as public benchmark

**Week 5: Full experiment**
- Run RUPS-300 × 5 models × 3 conditions = 4,500 API calls
- Auto-annotate with GPT-4o judge
- Generate paper tables and figures



In [ ]:
# Extract the 5 universal failure problems for Section 7 case studies
universal = ['HR-F4-027','FIN-F1-029','FIN-F3-004','FIN-F3-006','FIN-F5-006']

df_cases = pd.DataFrame(results/judged_results)
prob_map  = {p['id']: p for p in problems}

for pid in universal:
    prob = prob_map.get(pid)
    if not prob:
        continue
    print(f"\n{'='*65}")
    print(f"CASE STUDY: {pid}")
    print(f"Failure mode: {prob['failure_mode']} — {prob['failure_name']}")
    print(f"Trigger: {prob['structural_trigger']}")
    print(f"\nContext (first 300 chars):\n{prob['context'][:300]}")
    print(f"\nQuestion:\n{prob['question']}")
    print(f"\nGold answer (first 200 chars):\n{prob['gold_answer'][:200]}")

    # Show one baseline response per model
    rows = df_cases[(df_cases['id']==pid) & (df_cases['condition']=='baseline')]
    print(f"\nModel responses (baseline):")
    for _, row in rows.iterrows():
        print(f"  {row['model']:<22} score={row['judge_score']}  {str(row['judge_reason'])[:70]}")

In [ ]:
import pandas as pd, json, random

# Load judged results
df = pd.DataFrame(json.load(open('./results/results/judged_results_20260510_1247.json')))

# Sample 50 baseline responses — 10 per failure mode for balance
sample = df[df['condition']=='baseline'].groupby('failure_mode').apply(
    lambda x: x.sample(min(10, len(x)), random_state=42)
).reset_index(drop=True)

# Save as annotation sheet
sample[['id','domain','failure_mode','failure_name',
        'structural_trigger','response','judge_score',
        'judge_reason']].to_csv('./results/human_annotation_50.csv', index=False)

print(f'✅ {len(sample)} problems saved to human_annotation_50.csv')
print(sample['failure_mode'].value_counts())